# 🔢 Count Models — Poisson, Negative Binomial & ZINB
**ISLP Chapter 4 (Extended) · Pattern Recognition for the Rest of Us**

> When your outcome is a count — number of doctor visits, insurance claims, website clicks, product defects — linear regression is the wrong tool. Count models respect the discrete, non-negative, often skewed nature of count data.

---
### What you'll learn
- Why linear regression fails for count outcomes
- **Poisson regression** — the foundation of count modeling
- **Negative Binomial regression** — when counts are overdispersed
- **Zero-Inflated models (ZINB/ZIP)** — when there are too many zeros
- Interpreting Incidence Rate Ratios (IRR)
- Model diagnostics and choosing between models

### Real datasets
- **RAND Health Insurance Experiment** (mdvis) — outpatient doctor visits, n=20,190, real overdispersion
- **Committees dataset** — congressional committee activity, count outcome with excess zeros

### Time: ~70 minutes

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.datasets as smd
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':11
})
print("✓ All packages loaded — statsmodels includes real built-in datasets, no download needed")

In [ ]:
# ── Load real datasets (built into statsmodels — no internet required) ──────
# Dataset 1: RAND Health Insurance Experiment
# Famous experiment studying how health insurance cost-sharing affects medical utilization
randhie = smd.randhie.load_pandas().data.copy()

print("=== RAND Health Insurance Experiment ===")
print(f"n = {len(randhie):,} individuals")
print(f"\nVariables:")
print("  mdvis   = # outpatient doctor visits (our COUNT OUTCOME)")
print("  lncoins = log(coinsurance+1) — higher = more out-of-pocket cost")
print("  idp     = individual deductible plan (1=yes)")
print("  physlm  = physical limitation (1=yes)")
print("  disea   = number of chronic diseases")
print("  hlthg/f/p = self-rated health: good / fair / poor (omitted = excellent)")
print(f"\nOutcome summary:")
print(f"  Mean doctor visits: {randhie['mdvis'].mean():.2f}")
print(f"  Variance:           {randhie['mdvis'].var():.2f}")
print(f"  Var/Mean ratio:     {randhie['mdvis'].var()/randhie['mdvis'].mean():.2f}  ← >>1 means overdispersed")
print(f"  Zero rate:          {(randhie['mdvis']==0).mean():.1%}")
print(f"  Max visits:         {randhie['mdvis'].max()}")
randhie.head()

## 📐 Part 1 — Why Linear Regression Fails for Count Data

Count outcomes violate several key OLS assumptions:

1. **Non-negative** — OLS can predict negative counts (impossible)
2. **Integer-valued** — OLS predicts continuous values
3. **Heteroscedastic** — for Poisson: Var = Mean (variance grows with the mean)
4. **Right-skewed** — especially with rare events
5. **Multiplicative effects** — a predictor multiplies the expected count, not adds to it

The fix: model **log(μ)** as a linear function of predictors:
```
log(E[Y|X]) = β₀ + β₁X₁ + β₂X₂ + ...
```
This is the **log link** — ensures predictions are always positive and models multiplicative effects.

In [ ]:
# Show concretely why OLS fails on the real RAND data
from sklearn.linear_model import LinearRegression

X_ols = randhie[['lncoins','idp','physlm','disea','hlthg','hlthf','hlthp']].values
y     = randhie['mdvis'].values

ols   = LinearRegression().fit(X_ols, y)
y_hat = ols.predict(X_ols)
neg_pct = (y_hat < 0).mean() * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Distribution of doctor visits
ax = axes[0]
counts, bins = np.histogram(randhie['mdvis'], bins=range(0, 40))
ax.bar(range(0, 39), counts, color='#1e5fa8', edgecolor='white', alpha=0.85)
ax.set_title('Distribution of Doctor Visits\n(RAND data, n=20,190 real patients)')
ax.set_xlabel('# Outpatient Visits')
ax.set_ylabel('Count of Patients')
ax.axvline(randhie['mdvis'].mean(), color='#e85d20', lw=2,
           label=f"Mean={randhie['mdvis'].mean():.1f}")
ax.legend()

# 2. OLS predictions — including negatives
ax = axes[1]
ax.scatter(y_hat, y, alpha=0.07, s=8, color='#e85d20')
ax.axvline(0, color='#c0392b', lw=2.5, ls='--',
           label=f'{neg_pct:.1f}% predictions < 0 (impossible!)')
ax.set_xlabel('OLS Predicted Visits')
ax.set_ylabel('Actual Visits')
ax.set_title(f'OLS predicts {neg_pct:.1f}% negative values\n(impossible for a count outcome)')
ax.legend(fontsize=9)

# 3. Variance grows with mean — violates OLS homoscedasticity
ax = axes[2]
bins_q = pd.qcut(randhie['mdvis'], q=15, duplicates='drop')
grp = randhie.groupby(bins_q, observed=True)['mdvis'].agg(['mean', 'var']).dropna()
ax.scatter(grp['mean'], grp['var'], color='#1a7a45', s=70, zorder=3,
           label='Observed (quantile bins)')
x_line = np.linspace(0, grp['mean'].max(), 100)
ax.plot(x_line, x_line,              color='#1e5fa8', lw=2, ls='--', label='Poisson: Var=Mean')
ax.plot(x_line, x_line + x_line**2, color='#e85d20', lw=2, ls='--', label='NB: Var=Mean+Mean²')
ax.set_xlabel('Bin Mean')
ax.set_ylabel('Bin Variance')
actual_ratio = randhie['mdvis'].var() / randhie['mdvis'].mean()
ax.set_title(f'Variance >> Mean in real data\n(Var/Mean = {actual_ratio:.1f} — strongly overdispersed)')
ax.legend(fontsize=9)

plt.suptitle('Why OLS Fails for Count Data — RAND Health Insurance Data', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n📌 Real-world confirmation:")
print(f"   OLS predicts {neg_pct:.1f}% negative values — impossible for visit counts")
print(f"   Variance/Mean = {actual_ratio:.1f} — Poisson assumes 1.0, this data is 7x overdispersed")
print(f"   31.2% of patients have zero visits — excess zeros to model")


## 📊 Part 2 — Poisson Regression

**Model:**
```
log(E[mdvis]) = β₀ + β₁·lncoins + β₂·idp + β₃·physlm + β₄·disea + ...
```

**Incidence Rate Ratio (IRR) = exp(β):**
- IRR = 1.3 → predictor increases expected visits by 30%
- IRR = 0.8 → predictor decreases expected visits by 20%
- IRR = 1.0 → no effect

In [ ]:
# Fit Poisson regression
pois_model = smf.poisson(
    'mdvis ~ lncoins + idp + physlm + disea + hlthg + hlthf + hlthp',
    data=randhie
).fit(disp=False)

print(pois_model.summary())

In [ ]:
# Extract IRRs with confidence intervals
coefs = pois_model.params.drop('Intercept')
irr   = np.exp(coefs)
ci    = np.exp(pois_model.conf_int().drop('Intercept'))
pvals = pois_model.pvalues.drop('Intercept')

irr_df = pd.DataFrame({
    'Coef (β)':     coefs.round(4),
    'IRR':          irr.round(4),
    'CI Low':       ci[0].round(4),
    'CI High':      ci[1].round(4),
    'p-value':      pvals.round(4),
    'Sig':          pvals < 0.05
})
print("=== Poisson IRR Table ===\n")
print(irr_df.to_string())

print("\n📌 Plain-English interpretation:")
for name, row in irr_df.iterrows():
    if row['Sig']:
        direction = 'increases' if row['IRR'] > 1 else 'decreases'
        pct = abs(row['IRR'] - 1) * 100
        print(f"  {name:10s}: {direction} expected visits by {pct:.0f}%  (p={row['p-value']:.4f})")

In [ ]:
# IRR forest plot
fig, ax = plt.subplots(figsize=(9, 5))

y_pos  = range(len(irr_df))
colors = ['#1a7a45' if v > 1 else '#e85d20' for v in irr_df['IRR']]

ax.barh(list(irr_df.index), irr_df['IRR'] - 1, left=1,
        color=colors, alpha=0.75, edgecolor='white', height=0.55)
ax.errorbar(irr_df['IRR'], range(len(irr_df)),
            xerr=[irr_df['IRR'] - irr_df['CI Low'],
                  irr_df['CI High'] - irr_df['IRR']],
            fmt='none', color='#1a1d23', capsize=5, lw=2)

ax.axvline(1, color='black', lw=2, ls='--', label='IRR=1 (no effect)')
ax.set_xlabel('Incidence Rate Ratio (IRR)')
ax.set_title('Poisson Regression — Effect of Predictors on Doctor Visits\n'
             'Green = increases visits, Orange = decreases visits')
ax.legend()
plt.tight_layout()
plt.show()

print("\n📌 disea (chronic diseases) has the largest effect — each additional disease")
print(f"   multiplies expected visits by {np.exp(pois_model.params['disea']):.3f}")
print(f"   lncoins (cost) reduces visits — higher cost-sharing → fewer visits")

## ⚠️ Part 3 — Overdispersion: When Poisson Is Not Enough

**Poisson assumes Var(Y) = E(Y).** We already know this RAND data has Var/Mean ≈ 7.

Ignoring overdispersion causes:
- **Standard errors that are too small** → p-values too small → false discoveries
- Overconfident confidence intervals

**Negative Binomial** adds a dispersion parameter α:
```
Var(Y) = μ + α·μ²
```
When α→0: NB→Poisson. Larger α = more overdispersion.

In [ ]:
# Fit Negative Binomial
nb_model = smf.negativebinomial(
    'mdvis ~ lncoins + idp + physlm + disea + hlthg + hlthf + hlthp',
    data=randhie
).fit(disp=False)

print(nb_model.summary())
print(f"\nEstimated α (dispersion): {nb_model.params['alpha']:.4f}")
print(f"Implied Var(Y) at mean: μ + {nb_model.params['alpha']:.3f}·μ²")

In [ ]:
# The key comparison: Poisson vs NB standard errors
pois_se = pois_model.bse.drop('Intercept')
nb_se   = nb_model.bse.drop(['Intercept','alpha'])

se_compare = pd.DataFrame({
    'Poisson SE':   pois_se.round(5),
    'NegBin SE':    nb_se.round(5),
    'Ratio NB/Pois':  (nb_se / pois_se).round(3)
})
print("=== Standard Errors: Poisson vs Negative Binomial ===\n")
print(se_compare.to_string())

print(f"\n📌 NB standard errors are {(nb_se/pois_se).mean():.1f}x larger on average")
print("   Poisson was overconfident — it underestimated uncertainty")
print("   Some 'significant' Poisson predictors may not survive NB correction")

# AIC comparison
print(f"\n=== Model Fit (AIC — lower is better) ===")
print(f"  Poisson AIC:           {pois_model.aic:,.1f}")
print(f"  Negative Binomial AIC: {nb_model.aic:,.1f}  {'✓ Much better!' if nb_model.aic < pois_model.aic else ''}")
print(f"  ΔAIC = {pois_model.aic - nb_model.aic:,.0f}  (>10 = strong evidence for NB)")

## 0️⃣ Part 4 — Zero-Inflated Models

**31.2% of patients have zero visits.** How many does Poisson predict?

When there are **more zeros than any count model predicts**, we have **zero inflation** — suggesting two processes:
1. **Structural zeros** — people who would never visit a doctor in this period regardless of health (very healthy, avoidant, no access)
2. **Sampling zeros** — people who could visit but didn't during the observation window

**ZINB** = mixture of:
- A **logistic model** for P(structural zero)
- A **Negative Binomial** for the count among non-structural-zeros

In [ ]:
# Compare observed vs model-predicted zero rates
from scipy.stats import poisson as sp_poisson

mu_pois = pois_model.predict()
mu_nb   = nb_model.predict()
alpha_nb = nb_model.params['alpha']
r        = 1 / alpha_nb

pois_p0 = sp_poisson.pmf(0, mu_pois).mean()
nb_p0   = (r / (r + mu_nb) ** r).mean()
obs_p0  = (randhie['mdvis'] == 0).mean()

print(f"Observed zero rate:       {obs_p0:.3f} ({obs_p0*100:.1f}%)")
print(f"Poisson predicted zeros:  {pois_p0:.3f} ({pois_p0*100:.1f}%)")
print(f"NegBin predicted zeros:   {nb_p0:.3f} ({nb_p0*100:.1f}%)")
print(f"\nGap (observed - NegBin):  {obs_p0 - nb_p0:.3f}")
print("→ Even NB underpredicts zeros — zero inflation is present")

# Observed vs expected distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
k     = np.arange(0, 20)
obs   = [(randhie['mdvis'] == ki).mean() for ki in k]
p_p   = [sp_poisson.pmf(ki, mu_pois).mean() for ki in k]
p_nb  = [(stats.nbinom.pmf(ki, r, r/(r+mu_nb))).mean() for ki in k]

width = 0.28
ax.bar(k - width, obs, width, label='Observed',        color='#1a1d23', alpha=0.85)
ax.bar(k,         p_p, width, label='Poisson',         color='#e85d20', alpha=0.75)
ax.bar(k + width, p_nb,width, label='Neg Binomial',    color='#1e5fa8', alpha=0.75)
ax.set_xlabel('# Doctor Visits'); ax.set_ylabel('Proportion')
ax.set_title('Observed vs Model-Predicted Distribution\n'
             'Both Poisson and NB underpredict zeros in real data')
ax.legend(); ax.set_xlim(-0.5, 15)
plt.tight_layout()
plt.show()

In [ ]:
# Fit ZINB
X_count = sm.add_constant(randhie[['lncoins','idp','physlm','disea','hlthg','hlthf','hlthp']])
X_infl  = sm.add_constant(randhie[['lncoins','physlm','hlthg']])  # zero-inflation model

zinb = ZeroInflatedNegativeBinomialP(
    randhie['mdvis'], X_count, exog_infl=X_infl, p=2
).fit(method='bfgs', maxiter=500, disp=False)

print(zinb.summary())

In [ ]:
# Final model comparison
zinb_p0   = zinb.predict(which='prob-zero').mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Zero rate comparison
models_z    = ['Observed', 'Poisson', 'Neg Binomial', 'ZINB']
zero_rates  = [obs_p0, pois_p0, nb_p0, zinb_p0]
colors_bar  = ['#1a1d23', '#e85d20', '#1e5fa8', '#1a7a45']
bars = axes[0].bar(models_z, zero_rates, color=colors_bar, edgecolor='white', width=0.5)
for bar, val in zip(bars, zero_rates):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Predicted Zero Rate')
axes[0].set_title('Zero Rate: ZINB Matches Observed Best\n(RAND Doctor Visits data)')

# AIC comparison
try:
    aics = {'Poisson': pois_model.aic, 'Neg Binomial': nb_model.aic, 'ZINB': zinb.aic}
    colors_aic = ['#e85d20','#1e5fa8','#1a7a45']
    bars2 = axes[1].bar(list(aics.keys()), list(aics.values()), color=colors_aic,
                         edgecolor='white', width=0.5)
    for bar, val in zip(bars2, aics.values()):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                     f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[1].set_ylabel('AIC (lower = better fit)')
    axes[1].set_title('Model Selection by AIC\n(lower = better)')
except Exception as e:
    axes[1].text(0.5, 0.5, f'AIC comparison: see summaries above\n{e}',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=11)

plt.tight_layout()
plt.show()

print("\n=== Final Model Comparison ===")
print(f"  Poisson AIC:   {pois_model.aic:>12,.1f}  (ignores overdispersion)")
print(f"  NegBin AIC:    {nb_model.aic:>12,.1f}  (handles overdispersion)")
try:
    print(f"  ZINB AIC:      {zinb.aic:>12,.1f}  (handles overdispersion + excess zeros)")
    best = min([(pois_model.aic,'Poisson'),(nb_model.aic,'NegBin'),(zinb.aic,'ZINB')], key=lambda x: x[0])
    print(f"\n🏆 Best model: {best[1]} (lowest AIC = {best[0]:,.1f})")
except:
    print("  ZINB: see summary above")

## 🗺️ Part 5 — Choosing the Right Count Model

```
Is your outcome a non-negative integer (count)?
    ↓ Yes
Is Pearson χ²/df ≈ 1?
    ↓ Yes → POISSON    ↓ No (>>1) → overdispersed
                           Are there excess zeros?
                              ↓ No → NEG BINOMIAL
                              ↓ Yes → ZINB
```

| Model | Assumption | Use when |
|-------|-----------|----------|
| **Poisson** | Var = Mean | Counts with equidispersion (rare in practice) |
| **Neg Binomial** | Var = μ + αμ² | Overdispersed counts (most real data) |
| **ZIP** | Poisson + excess zeros | Zero-inflated, equidispersed counts |
| **ZINB** | NB + excess zeros | Overdispersed AND zero-inflated |

**Quick diagnostics:**
- `Pearson χ²/df >> 1` → overdispersion → use NB
- `observed P(Y=0) >> model P(Y=0)` → zero inflation → use ZIP/ZINB
- AIC to choose between candidates

## 💼 Exercise

**Task 1:** The RAND data has `lncoins` (cost-sharing). Fit a Negative Binomial and interpret the IRR for `lncoins` in plain English: what does higher out-of-pocket cost do to doctor visits?

**Task 2:** Load the `committees` dataset (`smd.committee.load_pandas().data`). The outcome `SUBS` = number of subcommittee assignments (count with 15% zeros). Fit Poisson, NB, and check for overdispersion.

**Task 3:** For the NB model on the RAND data, compute the predicted doctor visits for two hypothetical patients:
- Patient A: lncoins=4.6, idp=1, physlm=0, disea=0, hlthg=1, hlthf=0, hlthp=0 (healthy, high cost-sharing)
- Patient B: lncoins=0, idp=0, physlm=1, disea=5, hlthg=0, hlthf=0, hlthp=1 (poor health, no cost-sharing)

How many times more visits does Patient B have than Patient A?

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────

# Task 1: Interpret lncoins IRR
# YOUR CODE HERE

# Task 2: Committees dataset
import statsmodels.datasets as smd
committees = smd.committee.load_pandas().data
print("Committees dataset:", committees.shape)
print(committees.head())
# YOUR CODE HERE

# Task 3: Predict for two hypothetical patients
# YOUR CODE HERE

In [ ]:
# ── Submit your results ─────────────────────────────────────────────────────
GITHUB_USERNAME = ""   # ← your GitHub username

answers = {
    "q1": "",  # What does the Poisson assumption Var(Y)=E(Y) mean, and how do you test if it's violated?
    "q2": "",  # How do you interpret an IRR of 1.45 for a binary predictor 'smoker'?
    "q3": "",  # What is the difference between structural zeros and sampling zeros?
    "q4": "",  # When would you choose ZINB over Negative Binomial?
    "q5": "",  # Why are Poisson standard errors too small when data is overdispersed?
}

missing = [k for k,v in answers.items() if not str(v).strip()]
n_answered = len(answers) - len(missing)
print(f"{'='*50}")
print(f"Quiz: {n_answered}/{len(answers)} answered")
if not missing:
    print("✅ All answered! Save: File → Save a copy in GitHub")
else:
    print(f"❌ Still empty: {missing}")
print(f"{'='*50}")

## 📚 Further Reading

| Resource | What it covers |
|----------|---------------|
| [ISLP Ch 4.6](https://intro-stat-learning.github.io/ISLP/) | Poisson regression — count outcomes |
| [statsmodels GLM docs](https://www.statsmodels.org/stable/glm.html) | Poisson & NB in Python |
| [Cameron & Trivedi — Regression Analysis of Count Data](https://www.cambridge.org/core/books/regression-analysis-of-count-data/BBD7380C37A0FBF47E09D5E8EEDE14DC) | The definitive textbook |
| [Model Explainability notebook](model-explainability.ipynb) | SHAP for GLMs |
| [Logistic Regression notebook](logistic-regression.ipynb) | Binary outcome — the classification analogue |

---
---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*